<h1>SQLAlchemy Tutorial<h1/>

In [1]:
import sqlalchemy

In [2]:
sqlalchemy.__version__

'2.0.46'

In [3]:
from sqlalchemy import create_engine, text

In [4]:
engine = create_engine("sqlite+pysqlite:///:memory:", echo=True)

In [5]:
with engine.connect() as conn:
    result =  conn.execute(text("select 'hello world'"))
    print(result.all())

2026-01-24 22:48:49,080 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 22:48:49,081 INFO sqlalchemy.engine.Engine select 'hello world'
2026-01-24 22:48:49,082 INFO sqlalchemy.engine.Engine [generated in 0.00165s] ()
[('hello world',)]
2026-01-24 22:48:49,083 INFO sqlalchemy.engine.Engine ROLLBACK


In [6]:
# Commit as you go"
with engine.connect() as conn:
    conn.execute(text("CREATE TABLE some_table (x int, y int)"))
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"), 
        [{"x": 1, "y": 1}, {"x": 2, "y": 4}],
    )
    conn.commit()

2026-01-24 22:48:49,108 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 22:48:49,109 INFO sqlalchemy.engine.Engine CREATE TABLE some_table (x int, y int)
2026-01-24 22:48:49,111 INFO sqlalchemy.engine.Engine [generated in 0.00243s] ()
2026-01-24 22:48:49,114 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-24 22:48:49,116 INFO sqlalchemy.engine.Engine [generated in 0.00183s] [(1, 1), (2, 4)]
2026-01-24 22:48:49,117 INFO sqlalchemy.engine.Engine COMMIT


In [7]:
# begins once#
with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"),
        [{"x": 6, "y": 8}, {"x": 9, "y": 10}],
    )

2026-01-24 22:48:49,140 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 22:48:49,141 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-24 22:48:49,191 INFO sqlalchemy.engine.Engine [cached since 0.07753s ago] [(6, 8), (9, 10)]
2026-01-24 22:48:49,192 INFO sqlalchemy.engine.Engine COMMIT


In [8]:
# begins once#
with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (9, 10)")
            )

2026-01-24 22:48:49,220 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 22:48:49,222 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (9, 10)
2026-01-24 22:48:49,223 INFO sqlalchemy.engine.Engine [generated in 0.00132s] ()
2026-01-24 22:48:49,224 INFO sqlalchemy.engine.Engine COMMIT


In [9]:
# select statement#
with engine.begin() as conn:
    query_result = conn.execute(text("SELECT * FROM some_table"))
    print(query_result.all())

2026-01-24 22:48:49,250 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 22:48:49,252 INFO sqlalchemy.engine.Engine SELECT * FROM some_table
2026-01-24 22:48:49,254 INFO sqlalchemy.engine.Engine [generated in 0.00119s] ()
[(1, 1), (2, 4), (6, 8), (9, 10), (9, 10)]
2026-01-24 22:48:49,255 INFO sqlalchemy.engine.Engine COMMIT


In [10]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y FROM some_table"))
    for row in result:
        print(f"x: {row.x} y: {row.y}")

2026-01-24 22:48:49,284 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 22:48:49,285 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table
2026-01-24 22:48:49,286 INFO sqlalchemy.engine.Engine [generated in 0.00201s] ()
x: 1 y: 1
x: 2 y: 4
x: 6 y: 8
x: 9 y: 10
x: 9 y: 10
2026-01-24 22:48:49,289 INFO sqlalchemy.engine.Engine ROLLBACK


<h2>Sending Parameters<h2/>

In [11]:
# return value of y where its value is greater than a specific value)

with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y FROM some_table WHERE y > :y"), {"y": 8})
    for row in result:
        print(f"x: {row.x} y: {row.y}")

2026-01-24 22:48:49,317 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 22:48:49,319 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table WHERE y > ?
2026-01-24 22:48:49,320 INFO sqlalchemy.engine.Engine [generated in 0.00335s] (8,)
x: 9 y: 10
x: 9 y: 10
2026-01-24 22:48:49,322 INFO sqlalchemy.engine.Engine ROLLBACK


<h2>Sending Multiple Parameters<h2/>

In [12]:
# inserting multiple records in a sql statement

with engine.connect() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"), 
        [{"x": 11, "y": 12}, {"x": 13, "y": 14}]
    )
    conn.commit()

2026-01-24 22:48:49,345 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 22:48:49,346 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-24 22:48:49,347 INFO sqlalchemy.engine.Engine [cached since 0.2329s ago] [(11, 12), (13, 14)]
2026-01-24 22:48:49,348 INFO sqlalchemy.engine.Engine COMMIT


<h2>Executing with an ORM Session<h2/>

In [13]:
from sqlalchemy.orm import Session

In [14]:
stmt = text("SELECT x, y FROM some_table WHERE y > :y ORDER BY x, y")
with Session(engine) as session:
    result = session.execute(stmt, {"y": 6})
    for row in result:
        print(f"x: {row.x}, y: {row.y}")

2026-01-24 22:48:49,503 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 22:48:49,506 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table WHERE y > ? ORDER BY x, y
2026-01-24 22:48:49,507 INFO sqlalchemy.engine.Engine [generated in 0.00136s] (6,)
x: 6, y: 8
x: 9, y: 10
x: 9, y: 10
x: 11, y: 12
x: 13, y: 14
2026-01-24 22:48:49,510 INFO sqlalchemy.engine.Engine ROLLBACK


In [15]:
# commit #

with Session(engine) as session:
    result = session.execute(
        text("UPDATE some_table SET y=:y WHERE x=:x"),
        [{"x": 9, "y": 11},{"x": 13, "y": 15}],
    )
    session.commit()

2026-01-24 22:48:49,533 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 22:48:49,534 INFO sqlalchemy.engine.Engine UPDATE some_table SET y=? WHERE x=?
2026-01-24 22:48:49,535 INFO sqlalchemy.engine.Engine [generated in 0.00106s] [(11, 9), (15, 13)]
2026-01-24 22:48:49,537 INFO sqlalchemy.engine.Engine COMMIT


In [16]:
with Session(engine) as session:
    result = session.execute(
        text("UPDATE some_table SET y=:y WHERE x=:x"),
        [{"x": 9, "y": 11}, {"x": 13, "y": 15}]
    )
    session.commit()

2026-01-24 22:48:49,564 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 22:48:49,566 INFO sqlalchemy.engine.Engine UPDATE some_table SET y=? WHERE x=?
2026-01-24 22:48:49,567 INFO sqlalchemy.engine.Engine [cached since 0.03296s ago] [(11, 9), (15, 13)]
2026-01-24 22:48:49,568 INFO sqlalchemy.engine.Engine COMMIT


<h2>Setting up MetaData with Table objects<h2/>

In [17]:
from sqlalchemy import MetaData
metadata_obj = MetaData()

In [18]:
from sqlalchemy import Table, Column, Integer, String
user_table = Table(
    "user_account",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("name", String(30)),
    Column("fullname", String),
)

In [19]:
user_table.c.name

Column('name', String(length=30), table=<user_account>)

In [20]:
user_table.c.keys()

['id', 'name', 'fullname']

In [21]:
user_table.primary_key

PrimaryKeyConstraint(Column('id', Integer(), table=<user_account>, primary_key=True, nullable=False))

In [22]:
# create a second table
from sqlalchemy import ForeignKey

address_table = Table(
    "address",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("user_id", ForeignKey("user_account.id"), nullable=False),
    Column("email_address", String, nullable=False)
)

In [23]:
metadata_obj.create_all(engine)

2026-01-24 22:48:49,671 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 22:48:49,674 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2026-01-24 22:48:49,675 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-24 22:48:49,677 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("user_account")
2026-01-24 22:48:49,679 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-24 22:48:49,681 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("address")
2026-01-24 22:48:49,683 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-24 22:48:49,685 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("address")
2026-01-24 22:48:49,687 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-24 22:48:49,689 INFO sqlalchemy.engine.Engine 
CREATE TABLE user_account (
	id INTEGER NOT NULL, 
	name VARCHAR(30), 
	fullname VARCHAR, 
	PRIMARY KEY (id)
)


2026-01-24 22:48:49,691 INFO sqlalchemy.engine.Engine [no key 0.00119s] ()
2026-01-24 22:48:49,692 INFO sqlalchemy.engine.Engine 
C

<h2>Establishing a Declarative Base<h2/>

In [24]:
# create a new class that subclasses the SQLAlchemy DeclarativeBase class

from sqlalchemy.orm import DeclarativeBase

class Base(DeclarativeBase):
    pass

In [25]:
Base.metadata

MetaData()

In [26]:
Base.registry

In [27]:
from typing import List
from typing import Optional
from sqlalchemy.orm import Mapped
from sqlalchemy.orm import mapped_column
from sqlalchemy.orm import relationship

class User(Base):
    __tablename__ = "user_account"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(30))
    fullname: Mapped[Optional[str]]

    addresses: Mapped[List["Address"]] = relationship(back_populates="user")

    def __repr__(self) -> str:
        return f"User(id={self.id!r}, fullname={self.fullname!r})"
    

class Address(Base):
    __tablename__ = "address"

    id: Mapped[int] = mapped_column(primary_key=True)
    email_address: Mapped[str]
    user_id = mapped_column(ForeignKey("user_account.id"))

    user: Mapped[User] = relationship(back_populates="addresses")

    def __repr__(self) -> str:
        return f"Address(id={self.id!r}, email_address={self.email_address!r})"
        

<h2>Emitting DDL to the database from an ORM mapping<h2/>

In [28]:
Base.metadata.create_all(engine)

2026-01-24 22:48:49,795 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 22:48:49,796 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2026-01-24 22:48:49,796 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-24 22:48:49,797 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("address")
2026-01-24 22:48:49,799 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-24 22:48:49,800 INFO sqlalchemy.engine.Engine COMMIT


In [33]:
# import requests

# URL = "https://my.api.mockaroo.com/transaction.json?key=13f89f70"

# get_data = requests.get(URL)
# data = get_data.json()
# print(data)

[{'transactionID': 'c09d75e2-57fa-4aa6-bbff-e3175082183b', 'sender': 'Livia Calton', 'sender_location': 'Lagos', 'beneficiary': 'Bathsheba Malpass', 'beneficiary_location': 'Asaba', 'transaction_amount': '$29652.26', 'transaction_type': 'bill_payment', 'transaction_date': '7/7/2025', 'transaction_status': 'successful'}, {'transactionID': '046af9b6-6fc6-4c60-a9cc-dd1a3dd34d8b', 'sender': 'Elia Fulham', 'sender_location': 'Ikeja', 'beneficiary': 'Randolf Bruford', 'beneficiary_location': 'FCT', 'transaction_amount': '$42903.67', 'transaction_type': 'bill_payment', 'transaction_date': '10/26/2025', 'transaction_status': 'failed'}, {'transactionID': '0c5c0954-91ff-4229-b7df-3358e30ffc9b', 'sender': 'Carlee Baggalley', 'sender_location': 'FCT', 'beneficiary': 'Xenos Byrne', 'beneficiary_location': 'Makurdi', 'transaction_amount': '$28200.51', 'transaction_type': 'purchase', 'transaction_date': '7/6/2025', 'transaction_status': 'successful'}, {'transactionID': '1b7c69c9-fc1b-4b3c-a254-462155

In [34]:
some_table = Table("some_table", metadata_obj, autoload_with=engine)

2026-01-25 02:24:39,528 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-25 02:24:39,531 INFO sqlalchemy.engine.Engine PRAGMA main.table_xinfo("some_table")
2026-01-25 02:24:39,533 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-25 02:24:39,541 INFO sqlalchemy.engine.Engine SELECT sql FROM  (SELECT * FROM sqlite_master UNION ALL   SELECT * FROM sqlite_temp_master) WHERE name = ? AND type in ('table', 'view')
2026-01-25 02:24:39,543 INFO sqlalchemy.engine.Engine [raw sql] ('some_table',)
2026-01-25 02:24:39,550 INFO sqlalchemy.engine.Engine PRAGMA main.foreign_key_list("some_table")
2026-01-25 02:24:39,551 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-25 02:24:39,553 INFO sqlalchemy.engine.Engine PRAGMA temp.foreign_key_list("some_table")
2026-01-25 02:24:39,554 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-25 02:24:39,556 INFO sqlalchemy.engine.Engine SELECT sql FROM  (SELECT * FROM sqlite_master UNION ALL   SELECT * FROM sqlite_temp_master) WHERE name = ? AND type i

In [35]:
some_table

Table('some_table', MetaData(), Column('x', INTEGER(), table=<some_table>), Column('y', INTEGER(), table=<some_table>), schema=None)